# Cadquery + dotbimpy example

In [ ]:
from dotbimpy import *
import plotly.graph_objects as go
import pyquaternion
import numpy as np
import cadquery
import uuid

In [ ]:
def cadquery_mesh_to_dotbim_mesh(cadquery_mesh, mesh_id):
    vertices, triangles = cadquery_mesh
    coordinates = []
    for i in vertices:
        coordinates.extend([i.x, i.y, i.z])
    indices = [item for sublist in triangles for item in sublist]
    return Mesh(mesh_id=mesh_id, coordinates=coordinates, indices=indices)

In [ ]:
def create_plotly_figure(file):
    geometries = []
    for i in file.elements:
        mesh = next((x for x in file.meshes if x.mesh_id == i.mesh_id), None)
        geometries.extend(convert_to_plotly_meshes_with_face_colors(mesh, element=i))

    layout = go.Layout(scene=dict(aspectmode='data'))
    figure = go.Figure(data=[], layout=layout)
    for i in geometries:
        figure.add_trace(i)

    return figure

def convert_to_plotly(mesh, element):
        color_hex = '#%02x%02x%02x' % (element.color.r, element.color.g, element.color.b)
        opacity = element.color.a / 255

        x, y, z = repack_mesh_vertices_to_xyz_lists(mesh, element)
        i, j, k = repack_mesh_indices_to_ijk_lists(mesh)

        return go.Mesh3d(x=x, y=y, z=z, i=i, j=j, k=k, color=color_hex, opacity=opacity, name=element.type,
                         showscale=True)

def convert_to_plotly_meshes_with_face_colors(mesh, element):
    if not element.check_if_has_face_colors():
        return [convert_to_plotly(mesh, element)]
    else:
        plotly_meshes = []
        face_colors_counter = 0
        indices_counter = 0
        while face_colors_counter < len(element.face_colors):
            color_hex = '#%02x%02x%02x' % (element.face_colors[face_colors_counter],
                                            element.face_colors[face_colors_counter + 1],
                                            element.face_colors[face_colors_counter + 2])
            opacity = element.face_colors[face_colors_counter + 3] / 255

            i = [mesh.indices[indices_counter]]
            j = [mesh.indices[indices_counter + 1]]
            k = [mesh.indices[indices_counter + 2]]
            x, y, z = repack_mesh_vertices_to_xyz_lists(mesh, element)

            plotly_meshes.append(go.Mesh3d(x=x, y=y, z=z, i=i, j=j, k=k, color=color_hex, opacity=opacity,
                                            name=element.type, showscale=True))

            face_colors_counter += 4
            indices_counter += 3

        return plotly_meshes

def repack_mesh_indices_to_ijk_lists(mesh):
        i = []
        j = []
        k = []
        counter = 0
        while counter < len(mesh.indices):
            i.append(mesh.indices[counter])
            j.append(mesh.indices[counter + 1])
            k.append(mesh.indices[counter + 2])
            counter += 3

        return i, j, k

def repack_mesh_vertices_to_xyz_lists(mesh, element):

    x = []
    y = []
    z = []
    counter = 0
    while counter < len(mesh.coordinates):
        point = np.array([
            mesh.coordinates[counter],
            mesh.coordinates[counter + 1],
            mesh.coordinates[counter + 2]])

        rotation = pyquaternion.Quaternion(
            a=element.rotation.qw,
            b=element.rotation.qx,
            c=element.rotation.qy,
            d=element.rotation.qz)

        point_rotated = rotation.rotate(point)

        x.append(point_rotated[0] + element.vector.x)
        y.append(point_rotated[1] + element.vector.y)
        z.append(point_rotated[2] + element.vector.z)
        counter += 3

    return x, y, z

In [ ]:
meshes = []
elements = []

(L,H,W,t) = (6.000, 0.300, 0.200, 0.010)
pts = [(0,H/2.0),(W/2.0,H/2.0),(W/2.0,(H/2.0 - t)),(t/2.0,(H/2.0-t)),(t/2.0,(t - H/2.0)),(W/2.0,(t -H/2.0)),(W/2.0,H/-2.0),(0,H/-2.0)]
beam_workplane = cadquery.Workplane("XZ").polyline(pts).mirrorY().extrude(L)
beam_mesh_cq = beam_workplane.val().tessellate(0.1)
meshes.append(cadquery_mesh_to_dotbim_mesh(beam_mesh_cq, 0))

(number_of_beams,spacing,wall_height) = (5,2.000,3.000)
for i in range(number_of_beams):
    elements.append(Element(mesh_id=0,
                    vector=Vector(x=i*spacing, y=0, z=wall_height+H/2),
                    guid=str(uuid.uuid4()),
                    info={"Material": "Steel"},
                    rotation=Rotation(qx=0, qy=0, qz=0, qw=1.0),
                    type="Beam",
                    color=Color(r=0, g=0, b=255, a=255)))

wall_workplane = cadquery.Workplane("front").moveTo((spacing*(number_of_beams-1))/2.0,0).box(spacing*(number_of_beams-1)+W, 0.200, wall_height)
wall_mesh_cq = wall_workplane.val().tessellate(0.1)
meshes.append(cadquery_mesh_to_dotbim_mesh(wall_mesh_cq, 1))

for i in range(2):
    elements.append(Element(mesh_id=1,
                    vector=Vector(x=0, y=-i*L, z=wall_height/2.0),
                    guid=str(uuid.uuid4()),
                    info={"Material": "Concrete"},
                    rotation=Rotation(qx=0, qy=0, qz=0, qw=1.0),
                    type="Wall",
                    color=Color(r=255, g=215, b=0, a=255)))


file_info = {"Author": "John Doe"}
file = File(schema_version="1.0.0", meshes=meshes, elements=elements, info=file_info)
# View file
figure = create_plotly_figure(file)
figure.show()
# file.save("WallsWithBeams.bim")